# WebSight → формат контракта (drafting) — интерактивно

Пошаговый конвертер для отладки/просмотра. **Вся логика — в `convert_lib.py`**
(единый источник правды), здесь только оркестрация + настройки прогона.
Для массовой генерации на многих ядрах — `convert_parallel.py` (см. `README.md` + `Dockerfile`).

**Профили:** смоук (быстро, без браузера: `APPLY_PLACEHOLDERS=False`, `PRECOMPILE_TAILWIND=False`)
и production (self-contained: плейсхолдеры + precompile + ре-рендер; нужны `playwright`, `pytailwindcss`).

## 1. Настройки прогона

In [ ]:
import os, itertools, hashlib
from collections import Counter
from bs4 import BeautifulSoup
from datasets import load_dataset, Dataset, load_from_disk
import convert_lib as cl          # вся логика тут

# --- источник / объём ---
DATASET      = "HuggingFaceM4/WebSight"   # v0.2 (Tailwind); отладка: "mrm8488/WebSight_70k" (v0.1)
SPLIT        = "train"
TARGET_COUNT = 300                        # сколько ПРИНЯТЫХ сэмплов собрать
MAX_SCAN     = 20000
OUT_DIR      = os.path.abspath("./websight_drafting_pilot")

# --- профиль гигиены ---
APPLY_PLACEHOLDERS  = True    # True → серые плейсхолдеры + ре-рендер (нужны playwright+chromium)
PRECOMPILE_TAILWIND = True    # True → CDN-Tailwind вкомпилировать в <style> (нужен pytailwindcss)
DROP_IMG            = True    # смоук без плейсхолдеров: выкидывать сэмплы с <img>

# --- размер (только смоук; в production картинка перерисовывается) ---
TARGET_SIZE = None            # None = авто (доминирующий нативный); в production не используется
SIZE_MODE   = "filter"        # "filter" | "pad"

# --- прочее ---
DECONTAM_DOMAINS = set()      # блоклист доменов eval-бенчей (для синтетики WebSight пусто)
NEAR_DUP_HAMMING = None       # None = выкл; напр. 4
MAX_TOKENS       = None       # напр. 6144; None = без фильтра по длине кода

_tokenizer = None
if MAX_TOKENS is not None:
    from transformers import AutoTokenizer
    _tokenizer = AutoTokenizer.from_pretrained(cl.TOKENIZER_ID_DEFAULT)
print("out:", OUT_DIR, "| источник:", DATASET, "| profile:",
      "production" if (APPLY_PLACEHOLDERS and PRECOMPILE_TAILWIND) else "смоук")

## 2. Проба размера + фабрика стрима

In [ ]:
def fresh_stream():
    return load_dataset(DATASET, split=SPLIT, streaming=True)

# авто-размер нужен только смоуку (в production рендерим сами)
if TARGET_SIZE is None and not APPLY_PLACEHOLDERS:
    _sizes = Counter()
    for r in itertools.islice(fresh_stream(), 400):
        im = r.get("image")
        if im is not None:
            _sizes[im.size] += 1
    TARGET_SIZE = _sizes.most_common(1)[0][0]
    print(f"авто TARGET_SIZE = {TARGET_SIZE} (доминирует {_sizes[TARGET_SIZE]}/{sum(_sizes.values())})")
print("ключи примера:", list(next(iter(fresh_stream())).keys()))

## 3. Сборка (count-driven, ленивый стрим)

In [ ]:
rows, seen, _hashes = [], set(), []
skip = dict(empty=0, dup=0, img=0, size=0, long=0, decontam=0, neardup=0)
scanned = 0

for r in fresh_stream():
    if len(rows) >= TARGET_COUNT or scanned >= MAX_SCAN:
        break
    scanned += 1
    html = (r.get("text") or r.get("html") or "").strip()
    img  = r.get("image")
    if not html or img is None:
        skip["empty"] += 1; continue

    if DECONTAM_DOMAINS and (cl.page_domains(html) & DECONTAM_DOMAINS):
        skip["decontam"] += 1; continue

    if not APPLY_PLACEHOLDERS:                       # смоук: фиксируем один размер
        if SIZE_MODE == "filter" and img.size != TARGET_SIZE:
            skip["size"] += 1; continue
        elif SIZE_MODE == "pad":
            img = cl.fit_to_size(img, TARGET_SIZE)

    if NEAR_DUP_HAMMING is not None:
        h = cl.ahash(img)
        if any(cl.hamming(h, o) <= NEAR_DUP_HAMMING for o in _hashes):
            skip["neardup"] += 1; continue
        _hashes.append(h)

    if APPLY_PLACEHOLDERS:
        html, _ = cl.replace_images_with_placeholder(html)
    elif DROP_IMG and BeautifulSoup(html, "html.parser").find("img") is not None:
        skip["img"] += 1; continue

    if PRECOMPILE_TAILWIND:
        html = cl.precompile_tailwind(html)
    if APPLY_PLACEHOLDERS:                           # скрин перерисовать из финального html
        img = cl.render_threaded(html, cl.RENDER_WIDTH)

    if MAX_TOKENS is not None and cl.count_tokens(html, _tokenizer) > MAX_TOKENS:
        skip["long"] += 1; continue

    hh = hashlib.sha1(html.encode("utf-8")).hexdigest()
    if hh in seen:
        skip["dup"] += 1; continue
    seen.add(hh)
    rows.append({"task_type": "drafting", "images": [img.convert("RGB")],
                 "current_html": "", "target_html": html, "instruction": ""})

print(f"собрано: {len(rows)}/{TARGET_COUNT} (просмотрено {scanned}) | пропущено: {skip}")
if APPLY_PLACEHOLDERS:
    cl.close_renderer()

## 4. Сохранение + приёмка (контракт §6)

In [ ]:
assert rows, "rows пуст — проверь профиль/фильтры"
ds = Dataset.from_list(rows, features=cl.FEATURES)
ds.save_to_disk(OUT_DIR)

ds2 = load_from_disk(OUT_DIR)
assert ds2.column_names == ["task_type","images","current_html","target_html","instruction"]
sizes = {tuple(s["images"][0].size) for s in ds2}
if APPLY_PLACEHOLDERS:
    assert {w for w,_ in sizes} == {cl.RENDER_WIDTH}      # full_page: ширина фикс, высота плавает
else:
    assert sizes == {TARGET_SIZE}
import re as _re
n_cdn = sum(bool(_re.search(r'cdn[^"]*tailwind', s["target_html"], _re.I)) for s in ds2)
print(f"rows: {len(ds2)} | размеры: {sorted(sizes)[:3]}... | Tailwind-CDN: {n_cdn}")
print("статус:", "PRODUCTION (self-contained)" if (APPLY_PLACEHOLDERS and PRECOMPILE_TAILWIND and n_cdn==0) else "СМОУК")
ds2[0]["images"][0]

## 5. Токен-отчёт (код + картинка) + handoff SFT

In [ ]:
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(cl.TOKENIZER_ID_DEFAULT)
pairs = [(cl.count_tokens(s["target_html"], _tok), cl.qwen_image_tokens(*s["images"][0].size)) for s in ds2]
codes = sorted(c for c,_ in pairs); imgs = sorted(m for _,m in pairs); tot = sorted(c+m for c,m in pairs)
def _q(a,x): return a[min(len(a)-1, int(len(a)*x))]
IMAGE_TOKEN_BUDGET = _q(imgs, .99)
MAX_LENGTH = cl.recommend_max_length(codes, quantile=0.99, round_to=64, image_token_budget=IMAGE_TOKEN_BUDGET)
print(f"код:      median={codes[len(codes)//2]}, p99={_q(codes,.99)}")
print(f"картинка: median={imgs[len(imgs)//2]}, p99={IMAGE_TOKEN_BUDGET}")
print(f"всего:    median={tot[len(tot)//2]}, p99={_q(tot,.99)}")
print(f"\n=== ПЕРЕДАЧА SFT ===\nпуть: {OUT_DIR}\nзагрузка: load_from_disk(<путь>)")
print(f"max_length: {MAX_LENGTH} (код p99 + картинка p99)")
print(f"монтирование (§7): -v {OUT_DIR}:/data")